In [1]:
# Core imports
import os, sys, asyncio, yaml
from dotenv import load_dotenv
from pydantic import BaseModel
from contextlib import asynccontextmanager
# Project modules (assumes this notebook lives in src/mcp_client/)
from client import MCPClient
import json
import requests

In [2]:
import logging
import sys
from typing import Optional

DEFAULT_FORMAT = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

def get_logger(
    name: str,
    log_file: Optional[str] = None,
    level: int = logging.INFO,
    console_level: Optional[int] = None,
    fmt: str = DEFAULT_FORMAT,
    propagate: bool = False,
):
    """Return a configured logger.

    Ensures we don't attach duplicate handlers if called multiple times.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.propagate = propagate

    formatter = logging.Formatter(fmt)

    # Add/ensure file handler
    if log_file:
        if not any(isinstance(h, logging.FileHandler) and getattr(h, 'baseFilename', None) and h.baseFilename.endswith(log_file) for h in logger.handlers):
            fh = logging.FileHandler(log_file)
            fh.setLevel(level)
            fh.setFormatter(formatter)
            logger.addHandler(fh)

    # Add/ensure console handler
    if console_level is None:
        console_level = level
    if not any(isinstance(h, logging.StreamHandler) and getattr(h, 'stream', None) is sys.stdout for h in logger.handlers):
        ch = logging.StreamHandler(sys.stdout)
        ch.setLevel(console_level)
        ch.setFormatter(formatter)
        logger.addHandler(ch)

    return logger

In [3]:
load_dotenv()
api_key = os.getenv("OPENROUTER_API_KEY", "")

logger = get_logger("mcp-client", log_file="mcp_client.log", level=20, console_level=20)

from pathlib import Path
file_path = Path("/home/vmadmin/intent/src/test/teste.csv")

In [ ]:
# Create a client, connect, and process an intent (returns an OpenAI Responses API object)
client = MCPClient(logger=logger, rapp="http://10.233.7.113:8090", file_path= file_path)
connected = await client.connect_to_server("http://127.0.0.1:8000/sse")
await client.set_llm(api_key=api_key, llm_name="qwen",llm_model="qwen/qwen3-30b-a3b-instruct-2507")
if not connected:
    raise RuntimeError("Failed to connect to MCP server")

# # 'response' is an object (not a dict) with an 'output' attribute (a list of bl1ocks)
response = await client.process_intent(
   "Create a slice to support video journalists transmitting live footage from a downtown area, allowing up to 40 devices with downstream rates of 120 Mbps per device and a packet delay budget of 15 ms to keep streams smooth."
)

2026-09-21 21:41:26,461 - mcp-client - INFO - Attempting to connect to server at http://127.0.0.1:8000/sse.
2026-09-21 21:41:26,580 - mcp-client - INFO - Requesting MCP tools from the server.
2026-09-21 21:41:26,587 - mcp-client - INFO - Successfully connected to server. Available tools: ['create_session', 'get_session', 'delete_session', 'ping_api']
2026-09-21 21:41:26,587 - mcp-client - INFO - Setting llm qwen - model qwen/qwen3-30b-a3b-instruct-2507
2026-09-21 21:41:26,588 - mcp-client - INFO - Requesting MCP tools from the server.
2026-09-21 21:41:26,611 - mcp-client - INFO - LLM configuration completed successfully.
2026-09-21 21:41:26,611 - mcp-client - INFO - Calling LLM: qwen
2026-09-21 21:41:30,656 - mcp-client - INFO - Assistant response: ChatCompletion(id='gen-1790041286-jX4QA6uZ5OO4cSMGtXYh', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_c

Error in sse_reader
Traceback (most recent call last):
  File "/home/vmadmin/intent/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/home/vmadmin/intent/.venv/lib/python3.12/site-packages/httpx/_transports/default.py", line 271, in __aiter__
    async for part in self._httpcore_stream:
  File "/home/vmadmin/intent/.venv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 407, in __aiter__
    raise exc from None
  File "/home/vmadmin/intent/.venv/lib/python3.12/site-packages/httpcore/_async/connection_pool.py", line 403, in __aiter__
    async for part in self._stream:
  File "/home/vmadmin/intent/.venv/lib/python3.12/site-packages/httpcore/_async/http11.py", line 342, in __aiter__
    raise exc
  File "/home/vmadmin/intent/.venv/lib/python3.12/site-packages/httpcore/_async/http11.py", line 334, in __aiter__
    async for chunk in self._connection._receive_response_body(**kwargs):
  File "/home/

In [5]:
print(response)

{"status":"Policy created successfully","code":200,"message":"Policy created successfully (policy_id=4a79c62c-c546-408e-bfff-6531cf13e5e5)"}
